# NB32 — KANSER Panel Optimization: Training Pool Sweep + Finetune Fix

**Tarih:** 2026-06-23  
**Hedef:** NB31 E6_alpha_0.5 Boot-F1=0.7201 platosunu kırmak  
**Strateji:** 7 model × 10 training pool varyasyonu + NB16 finetune düzeltmesi + stacking + ensemble

## Deneyler
| Exp | Açıklama |
|-----|----------|
| 0 | Finetune düzeltme doğrulama (NB31 bugfix) |
| 1 | Tam Pool × Model Sweep (7 model × 10 pool) |
| 2 | FE etkileşim testi |
| 3 | Stacking (en iyi config) |
| 4 | Ensemble sweep (Exp3 + NB31 E6) |

## Training Pool Varyasyonları
| Pool | Açıklama | ~Boyut |
|------|----------|--------|
| P0 | KANSER-only | 192 |
| P1 | MASTER balanced 625/625 | 1250 |
| P2 | P1 + KANSER train | 1442 |
| P3 | COMBINED ham | 3414 |
| P4 | COMBINED + KANSER | 3606 |
| P5 | COMBINED %80B/%20P | ~1081 |
| P6 | MASTER undersample | ~388 |
| P7 | P6 + KANSER | ~580 |
| P8 | COMBINED balanced %50/%50 | ~1730 |
| P9 | COMBINED %60B/%40P | ~1442 |

In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, json, gc
from copy import deepcopy
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score, matthews_corrcoef,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier

import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print('[UYARI] catboost bulunamadi')

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'teknofest_model'))
sys.path.insert(0, os.path.abspath('..'))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR

# --- Sabitler ---
PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
N_OOF_FOLDS = 5
BOOT_SEED = 123
PANEL_SPLIT_FRAC = 0.50

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v17_pretrain_distribution')
os.makedirs(RESULTS_DIR, exist_ok=True)
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f'SEED={SEED}, PROJECT_ROOT={PROJECT_ROOT}')
print(f'Results -> {RESULTS_DIR}')

SEED=42, PROJECT_ROOT=/Users/tefe/teknofest_model/teknofest_model
Results -> /Users/tefe/teknofest_model/teknofest_model/results/v17_pretrain_distribution


In [2]:
# Cell 2: Veri Yukleme + Sutun Temizligi
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

data_dir = os.path.join(PROJECT_ROOT, 'data', 'real_data')
df_master = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_MASTER.csv'))
df_kanser = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_KANSER.csv'))
df_cftr = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_CFTR.csv'))
df_pah = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_PAH.csv'))

print(f'MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})')
print(f'KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})')
print(f'CFTR:   {df_cftr.shape} (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})')
print(f'PAH:    {df_pah.shape} (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})')

# COMBINED: MASTER+PAH+CFTR (KANSER HARIC)
df_combined_raw = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)
print(f'\nCOMBINED: {df_combined_raw.shape} (pos={df_combined_raw[TARGET].sum()}, neg={(df_combined_raw[TARGET]==0).sum()})')

# Cross-panel birebir-ayni satir drop
feat_cols_raw = [c for c in df_kanser.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_kanser, df_master, feat_cols_raw, TARGET)
if dup_ids:
    df_kanser = df_kanser[~df_kanser[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f'KANSER: {len(dup_ids)} birebir-ayni satir drop -> {df_kanser.shape}')
else:
    print('KANSER: birebir-ayni satir yok')

# Sutun temizligi
constant_cols = [c for c in feat_cols_raw if df_master[c].nunique(dropna=False) <= 1]

def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, feat_cols_raw)
drop_cols = set(constant_cols) | dup_drop
keep_cols = [c for c in feat_cols_raw if c not in drop_cols]
print(f'Constant: {len(constant_cols)}, Dup pairs: {len(dup_pairs)} -> drop {len(dup_drop)}')
print(f'Toplam drop: {len(drop_cols)}, Kalan feature: {len(keep_cols)}')

for _df in [df_master, df_kanser, df_cftr, df_pah]:
    for c in drop_cols:
        if c in _df.columns:
            _df.drop(columns=[c], inplace=True)

df_combined = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)
print(f'\nFinal: MASTER={df_master.shape}, COMBINED={df_combined.shape}, KANSER={df_kanser.shape}')

# KANSER 50/50 split (NB31 ile ayni)
def panel_5050_split(df):
    pos = df[df[TARGET]==1].sample(frac=1.0, random_state=SEED)
    neg = df[df[TARGET]==0].sample(frac=1.0, random_state=SEED)
    npos = int(round(len(pos) * PANEL_SPLIT_FRAC))
    nneg = int(round(len(neg) * PANEL_SPLIT_FRAC))
    tr = pd.concat([pos.iloc[:npos], neg.iloc[:nneg]]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    te = pd.concat([pos.iloc[npos:], neg.iloc[nneg:]]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return tr, te

kanser_train, kanser_test = panel_5050_split(df_kanser)
y_kanser_train = kanser_train[TARGET].values
y_kanser_test = kanser_test[TARGET].values
print(f'\nKANSER train: {kanser_train.shape} (pos={y_kanser_train.sum()}, neg={(y_kanser_train==0).sum()})')
print(f'KANSER test:  {kanser_test.shape} (pos={y_kanser_test.sum()}, neg={(y_kanser_test==0).sum()})')
assert y_kanser_test.sum() > 0 and (y_kanser_test==0).sum() > 0, 'Split hatasi!'

NUM_COLS_BASE = [c for c in keep_cols if df_master[c].dtype != 'object']
CAT_COLS_BASE = [c for c in keep_cols if df_master[c].dtype == 'object']
print(f'Numeric: {len(NUM_COLS_BASE)}, Categorical: {len(CAT_COLS_BASE)}')

MASTER: (2931, 353) (pos=2149, neg=782)
KANSER: (388, 353) (pos=268, neg=120)
CFTR:   (111, 353) (pos=90, neg=21)
PAH:    (372, 353) (pos=310, neg=62)

COMBINED: (3414, 353) (pos=2549, neg=865)
KANSER: 3 birebir-ayni satir drop -> (385, 353)
Constant: 0, Dup pairs: 58 -> drop 58
Toplam drop: 58, Kalan feature: 293

Final: MASTER=(2931, 295), COMBINED=(3414, 295), KANSER=(385, 295)

KANSER train: (192, 295) (pos=132, neg=60)
KANSER test:  (193, 295) (pos=133, neg=60)
Numeric: 286, Categorical: 7


In [3]:
# Cell 3: M3 Preprocessing
AA_UNK = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
HIGH_MISS_THR = 0.50

def fit_preprocessor(train_df, keep_cols=keep_cols, target=TARGET):
    """Train uzerinde fit: median, label encoder, high-missing tespiti."""
    X = train_df[keep_cols].copy()
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    medians = X[num_cols].median()
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians,
        "le_maps": le_maps, "keep_cols": keep_cols
    }

def transform_X(df, prep):
    """Preprocessor uygula, is_missing flagleri ekle."""
    kc = prep["keep_cols"]
    X = df[kc].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    return X

print("M3 Preprocessing hazir.")

M3 Preprocessing hazir.


In [4]:
# Cell 4: Degerlendirme Altyapisi

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y); prob = np.asarray(prob)
    neg = np.where(y == 0)[0]; pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    return float(max(thr_scores, key=thr_scores.get))

def train_metrics_at(y_train, p_train, thr):
    yp = (p_train >= thr).astype(int)
    return {
        "train_f1": float(_f1_pos(y_train, yp)),
        "train_mcc": float(matthews_corrcoef(y_train, yp)),
        "train_prec": float(precision_score(y_train, yp, pos_label=1, zero_division=0)),
        "train_rec": float(recall_score(y_train, yp, pos_label=1, zero_division=0))
    }

def eval_model(label, y_test, p_test, y_train, p_train):
    thr = select_threshold_8020_robust(y_train, p_train)
    boot = bootstrap_8020(y_test, p_test, thr)
    y_pred = (p_test >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
    auprc = average_precision_score(y_test, p_test) if len(np.unique(y_test)) > 1 else 0.0
    prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    tr_met = train_metrics_at(y_train, p_train, thr)
    return {
        "boot_f1": boot["mean"], "boot_std": boot["std"],
        "boot_lo": boot["lo"], "boot_hi": boot["hi"],
        "auprc": auprc, "precision": prec, "recall": rec, "mcc": mcc,
        "fp": int(fp), "fn": int(fn), "tp": int(tp), "tn": int(tn),
        "thr": thr, **tr_met
    }

print("Degerlendirme altyapisi hazir.")

Degerlendirme altyapisi hazir.


In [5]:
# Cell 5: Tree Model Helpers

LGBM_PARAMS = {
    "n_estimators": 300, "num_leaves": 31, "learning_rate": 0.05,
    "min_child_samples": 20, "subsample": 0.8, "colsample_bytree": 0.8,
    "class_weight": "balanced",
    "random_state": SEED, "verbose": -1, "n_jobs": -1, "importance_type": "gain"
}
XGB_PARAMS = {
    "n_estimators": 300, "max_depth": 6, "learning_rate": 0.05,
    "subsample": 0.8, "colsample_bytree": 0.8,
    "random_state": SEED, "verbosity": 0, "n_jobs": -1
}
CB_PARAMS = {
    "iterations": 300, "depth": 6, "learning_rate": 0.05,
    "auto_class_weights": "Balanced",
    "random_seed": SEED, "verbose": 0
}

def _prep_tree(X_df):
    """Kategorikleri lgbm/xgb icin hazirla."""
    cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()
    Xn = X_df.copy()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, cat_cols, le_maps

def _apply_le(X_df, cat_cols, le_maps):
    Xn = X_df.copy()
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = le_maps[c]
        Xn[c] = Xn[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    return Xn

def oof_tree(model_type, X_train_df, y_train, X_test_df, n_splits=N_OOF_FOLDS):
    """Tree model OOF: train uzerinde oof + test uzerinde full-model proba."""
    X_tr, cat_cols, le_maps = _prep_tree(X_train_df)
    X_te = _apply_le(X_test_df, cat_cols, le_maps)

    # lgbm icin category type
    if model_type == "lgbm":
        for c in cat_cols:
            X_tr[c] = X_tr[c].astype("category")
            X_te[c] = pd.Categorical(X_te[c], categories=X_tr[c].cat.categories)

    oof = np.zeros(len(y_train))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for tri, vai in skf.split(X_tr, y_train):
        if model_type == "lgbm":
            m = LGBMClassifier(**LGBM_PARAMS)
            m.fit(X_tr.iloc[tri], y_train[tri], categorical_feature=cat_cols)
        elif model_type == "xgb":
            spw = float((y_train[tri]==0).sum()) / max(float((y_train[tri]==1).sum()), 1)
            m = XGBClassifier(**{**XGB_PARAMS, "scale_pos_weight": spw})
            m.fit(X_tr.iloc[tri], y_train[tri])
        elif model_type == "catboost":
            cat_idx = [list(X_tr.columns).index(c) for c in cat_cols]
            m = CatBoostClassifier(**CB_PARAMS)
            m.fit(X_tr.iloc[tri], y_train[tri], cat_features=cat_idx, silent=True)
        oof[vai] = m.predict_proba(X_tr.iloc[vai])[:, 1]

    # Full model
    if model_type == "lgbm":
        mf = LGBMClassifier(**LGBM_PARAMS)
        mf.fit(X_tr, y_train, categorical_feature=cat_cols)
    elif model_type == "xgb":
        spw = float((y_train==0).sum()) / max(float((y_train==1).sum()), 1)
        mf = XGBClassifier(**{**XGB_PARAMS, "scale_pos_weight": spw})
        mf.fit(X_tr, y_train)
    elif model_type == "catboost":
        cat_idx = [list(X_tr.columns).index(c) for c in cat_cols]
        mf = CatBoostClassifier(**CB_PARAMS)
        mf.fit(X_tr, y_train, cat_features=cat_idx, silent=True)

    test_proba = mf.predict_proba(X_te)[:, 1]
    return oof, test_proba

print("Tree model helpers hazir: lgbm, xgb, catboost")

Tree model helpers hazir: lgbm, xgb, catboost


In [6]:
# Cell 6: NN/DNN Models — NB16 _train_es recetesi (tum bugfix'ler dahil)

class SmallMLP(nn.Module):
    """NB16 tarzi: sabit genislik, parametrize n_layers, BatchNorm YOK."""
    def __init__(self, d, hidden=128, n_layers=2, dropout=0.4):
        super().__init__()
        layers = []; inp = d
        for _ in range(n_layers):
            layers += [nn.Linear(inp, hidden), nn.ReLU(), nn.Dropout(dropout)]
            inp = hidden
        layers += [nn.Linear(inp, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

class DeepMLP(nn.Module):
    """3 katman + residual, BatchNorm YOK."""
    def __init__(self, d, hidden=128, n_layers=3, dropout=0.4):
        super().__init__()
        self.input_proj = nn.Linear(d, hidden)
        self.blocks = nn.ModuleList()
        for _ in range(n_layers):
            self.blocks.append(nn.Sequential(
                nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout)
            ))
        self.output = nn.Linear(hidden, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout * 0.5)
    def forward(self, x):
        x = self.dropout(self.relu(self.input_proj(x)))
        for i, block in enumerate(self.blocks):
            residual = x
            x = block(x)
            if i % 2 == 1:
                x = x + residual
        return self.output(x).squeeze(-1)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, pos_weight=None):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.pos_weight = pos_weight
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        if self.pos_weight is not None:
            w = torch.where(targets >= 0.5, self.pos_weight, torch.ones_like(self.pos_weight))
            bce = bce * w
        p_t = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()

def _make_mlp(kind, d):
    """NB16 tarzi: nn=2 layer dp=0.4, dnn=3 layer dp=0.5."""
    hidden, nl, dp = (128, 2, 0.4) if kind in ("nn", "nn_ft") else (128, 3, 0.5)
    return SmallMLP(d, hidden, nl, dp)

def _train_es(model, X, y, lr=1e-3, max_epochs=60, patience=10):
    """NB16 _train_es birebir: F1-based ES, pos_weight, weight_decay=1e-3."""
    yv = np.asarray(y)
    strat = yv if min((yv==0).sum(), (yv==1).sum()) >= 2 else None
    tri, vai = train_test_split(np.arange(len(yv)), test_size=0.25,
                                random_state=SEED, stratify=strat)
    pw = torch.FloatTensor([(yv[tri]==0).sum() / max((yv[tri]==1).sum(), 1)])
    crit = FocalLoss(alpha=0.25, gamma=2.0, pos_weight=pw)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    Xt = X[tri]; yt = torch.FloatTensor(yv[tri].astype(np.float32))
    Xvl = X[vai]; yvl = yv[vai]
    n = len(Xt); bs = min(64, max(2, n - 1))
    best, state, pat = -1, None, 0
    for _ in range(max_epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            if len(idx) < 2:
                continue
            opt.zero_grad()
            loss = crit(model(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            vp = torch.sigmoid(model(Xvl)).numpy()
        f = _f1_pos(yvl, (vp >= 0.5).astype(int))
        if f > best:
            best, state, pat = f, deepcopy(model.state_dict()), 0
        else:
            pat += 1
            if pat >= patience:
                break
    if state is not None:
        model.load_state_dict(state)
    return model

def _nn_proba(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X)).numpy()

def _nn_encode(basis_df, prep):
    """Scaler+LE fit on basis_df. NB16 pattern."""
    X = transform_X(basis_df, prep)
    cat_cols = [c for c in X.columns if c in prep["cat_cols"]]
    le_maps = {}
    Xn = X.copy()
    for c in cat_cols:
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c].astype(str))
        le_maps[c] = {v: i for i, v in enumerate(le.classes_)}
    Xn = Xn.astype(np.float32)
    sc = StandardScaler().fit(Xn.values)
    return {"prep": prep, "cat_cols": cat_cols, "le_maps": le_maps,
            "scaler": sc, "columns": list(Xn.columns)}

def _nn_matrix(df, enc):
    """Transform df -> FloatTensor."""
    X = transform_X(df, enc["prep"])
    Xn = X.copy()
    for c in enc["cat_cols"]:
        Xn[c] = Xn[c].astype(str).map(enc["le_maps"][c]).fillna(-1)
    vals = Xn[enc["columns"]].astype(np.float32).values
    return torch.FloatTensor(enc["scaler"].transform(vals))

def oof_nn_scratch(kind, pool_df, test_df, prep, n_splits=N_OOF_FOLDS):
    """Scratch NN/DNN: NB16 _train_es ile OOF."""
    enc = _nn_encode(pool_df, prep)
    Xtr = _nn_matrix(pool_df, enc)
    Xte = _nn_matrix(test_df, enc)
    y = pool_df[TARGET].values
    oof = np.zeros(len(y))
    ns = min(n_splits, min((y==0).sum(), (y==1).sum()))
    ns = max(2, ns)
    skf = StratifiedKFold(n_splits=ns, shuffle=True, random_state=SEED)
    for tri, vai in skf.split(Xtr.numpy(), y):
        m = _make_mlp(kind, Xtr.shape[1])
        m = _train_es(m, Xtr[tri], y[tri])
        oof[vai] = _nn_proba(m, Xtr[vai])
    mfull = _make_mlp(kind, Xtr.shape[1])
    mfull = _train_es(mfull, Xtr, y)
    test_proba = _nn_proba(mfull, Xte)
    return oof, test_proba

def oof_nn_finetune(kind, pretrain_df, kanser_train_df, kanser_test_df, prep,
                    n_splits=N_OOF_FOLDS):
    """NB16-tarzi finetune: 4 bug duzeltmesi dahil.
    Bug#1: Tum katmanlar acik (freeze yok)
    Bug#2: Birlesik scaler (pretrain+panel)
    Bug#3: F1-based ES
    Bug#4: pretrain_df = pool varyasyonu
    """
    # Scaler basis = pretrain + kanser_train (Bug#2 fix)
    fit_basis = pd.concat([pretrain_df, kanser_train_df], ignore_index=True)
    enc = _nn_encode(fit_basis, prep)

    # Pretrain
    Xpre = _nn_matrix(pretrain_df, enc)
    ypre = pretrain_df[TARGET].values
    base_model = _make_mlp(kind, Xpre.shape[1])
    base_model = _train_es(base_model, Xpre, ypre, lr=1e-3, max_epochs=60, patience=10)

    # OOF finetune on kanser_train
    Xkan = _nn_matrix(kanser_train_df, enc)
    ykan = kanser_train_df[TARGET].values
    Xte = _nn_matrix(kanser_test_df, enc)
    oof = np.zeros(len(ykan))
    ns = min(n_splits, min((ykan==0).sum(), (ykan==1).sum()))
    ns = max(2, ns)
    skf = StratifiedKFold(n_splits=ns, shuffle=True, random_state=SEED)
    fold_models = []
    for tri, vai in skf.split(Xkan.numpy(), ykan):
        m = deepcopy(base_model)
        # Bug#1 fix: tum parametreler optimizer'da, freeze yok
        m = _train_es(m, Xkan[tri], ykan[tri], lr=1e-4, max_epochs=30, patience=8)
        oof[vai] = _nn_proba(m, Xkan[vai])
        fold_models.append(m)

    # Test: fold ortalamasi
    test_proba = np.zeros(len(Xte))
    for fm in fold_models:
        test_proba += _nn_proba(fm, Xte)
    test_proba /= len(fold_models)

    return oof, test_proba

print("NN/DNN modelleri hazir (NB16 recetesi, 4 bugfix dahil)")

NN/DNN modelleri hazir (NB16 recetesi, 4 bugfix dahil)


In [7]:
# Cell 7: Feature Engineering (NB16 Cell 4 — Grantham/BLOSUM62/stopgain)

_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}

def grantham(a, b):
    if a == b: return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a))

_B62_RAW = '''A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4'''
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for ri, line in enumerate(_B62_RAW.strip().split("\n")):
    toks = line.split()
    row_aa = toks[0][0]
    vals = [toks[0][1:]] + toks[1:]
    for ci, tok in enumerate(vals):
        col_aa = _ORDER[ri + ci]
        v = int(tok[1:] if tok[0].isalpha() else tok)
        _B62[(row_aa, col_aa)] = v; _B62[(col_aa, row_aa)] = v

def blosum62(a, b):
    return _B62.get((a, b), 0)

def add_fe(df):
    out = df.copy()
    a1 = out["AA_1"].astype("object"); a2 = out["AA_2"].astype("object")
    out["fe_aa_stopgain"] = (a2 == "*").astype(int)
    def _nonstd(v):
        return 0 if (isinstance(v, str) and v in STANDARD_AA) else 1
    out["fe_aa_nonstandard"] = (a1.map(_nonstd) | a2.map(_nonstd)).astype(int)
    def _gr(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return grantham(x, y)
        return -1
    def _bl(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return blosum62(x, y)
        return 0
    out["fe_grantham"] = out.apply(_gr, axis=1).astype(float)
    out["fe_blosum62"] = out.apply(_bl, axis=1).astype(float)
    return out

FE_NEW_COLS = ["fe_aa_stopgain", "fe_aa_nonstandard", "fe_grantham", "fe_blosum62"]
print(f"FE hazir. Yeni sutunlar: {FE_NEW_COLS}")

FE hazir. Yeni sutunlar: ['fe_aa_stopgain', 'fe_aa_nonstandard', 'fe_grantham', 'fe_blosum62']


In [8]:
# Cell 8: Training Pool Builder (10 varyasyon)

def build_pool(pool_name):
    """Training pool varyasyonlarini olustur. Returns (pool_df, description)."""
    rng = np.random.RandomState(SEED)

    if pool_name == "P0_KANSER_ONLY":
        return kanser_train.copy(), "KANSER train only (192)"

    elif pool_name == "P1_BALANCED_625":
        pos = df_master[df_master[TARGET]==1].sample(n=625, random_state=SEED)
        neg = df_master[df_master[TARGET]==0].sample(n=625, random_state=SEED)
        pool = pd.concat([pos, neg]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        return pool, "MASTER balanced 625P/625B (1250)"

    elif pool_name == "P2_KANSER_BAL625":
        pos = df_master[df_master[TARGET]==1].sample(n=625, random_state=SEED)
        neg = df_master[df_master[TARGET]==0].sample(n=625, random_state=SEED)
        master_core = pd.concat([pos, neg])
        pool = pd.concat([master_core, kanser_train], ignore_index=True).sample(
            frac=1.0, random_state=SEED).reset_index(drop=True)
        return pool, "MASTER 625/625 + KANSER train (~1442)"

    elif pool_name == "P3_COMBINED_RAW":
        return df_combined.copy(), f"COMBINED ham ({len(df_combined)})"

    elif pool_name == "P4_COMBINED_KANSER":
        pool = pd.concat([df_combined, kanser_train], ignore_index=True).reset_index(drop=True)
        return pool, f"COMBINED + KANSER train ({len(pool)})"

    elif pool_name == "P5_FINAL_8020":
        comb_neg = df_combined[df_combined[TARGET]==0]
        comb_pos = df_combined[df_combined[TARGET]==1]
        n_neg = len(comb_neg)
        n_pos = max(1, int(round(n_neg * 0.20 / 0.80)))
        pos_sample = comb_pos.sample(n=min(n_pos, len(comb_pos)), random_state=SEED)
        pool = pd.concat([comb_neg, pos_sample]).sample(
            frac=1.0, random_state=SEED).reset_index(drop=True)
        return pool, f"COMBINED %80B/%20P ({len(pool)})"

    elif pool_name == "P6_MASTER_UNDER":
        m_pos = df_master[df_master[TARGET]==1]
        m_neg = df_master[df_master[TARGET]==0]
        # KANSER oranina undersample: ~2.23:1 (268P/120B)
        n_neg = min(120, len(m_neg))
        n_pos = min(int(round(n_neg * 268.0 / 120.0)), len(m_pos))
        pool = pd.concat([
            m_pos.sample(n=n_pos, random_state=SEED),
            m_neg.sample(n=n_neg, random_state=SEED)
        ]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        return pool, f"MASTER undersample KANSER oraninda ({len(pool)})"

    elif pool_name == "P7_MASTER_UNDER_K":
        p6, _ = build_pool("P6_MASTER_UNDER")
        pool = pd.concat([p6, kanser_train], ignore_index=True).sample(
            frac=1.0, random_state=SEED).reset_index(drop=True)
        return pool, f"MASTER undersample + KANSER ({len(pool)})"

    elif pool_name == "P8_COMBINED_BAL":
        comb_neg = df_combined[df_combined[TARGET]==0]
        comb_pos = df_combined[df_combined[TARGET]==1]
        n_min = min(len(comb_neg), len(comb_pos))
        pool = pd.concat([
            comb_neg.sample(n=n_min, random_state=SEED),
            comb_pos.sample(n=n_min, random_state=SEED)
        ]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        return pool, f"COMBINED balanced %50/%50 ({len(pool)})"

    elif pool_name == "P9_REVERSE_6040":
        comb_neg = df_combined[df_combined[TARGET]==0]
        comb_pos = df_combined[df_combined[TARGET]==1]
        n_neg = len(comb_neg)
        n_pos = max(1, int(round(n_neg * 0.40 / 0.60)))
        pos_sample = comb_pos.sample(n=min(n_pos, len(comb_pos)), random_state=SEED)
        pool = pd.concat([comb_neg, pos_sample]).sample(
            frac=1.0, random_state=SEED).reset_index(drop=True)
        return pool, f"COMBINED %60B/%40P ({len(pool)})"

    else:
        raise ValueError(f"Bilinmeyen pool: {pool_name}")

POOL_NAMES = [
    "P0_KANSER_ONLY", "P1_BALANCED_625", "P2_KANSER_BAL625",
    "P3_COMBINED_RAW", "P4_COMBINED_KANSER", "P5_FINAL_8020",
    "P6_MASTER_UNDER", "P7_MASTER_UNDER_K", "P8_COMBINED_BAL",
    "P9_REVERSE_6040"
]

print("Pool varyasyonlari:")
for pn in POOL_NAMES:
    pdf, desc = build_pool(pn)
    pos = pdf[TARGET].sum(); neg = (pdf[TARGET]==0).sum()
    print(f"  {pn:25s}: {desc} [pos={pos}, neg={neg}, ratio={pos/(pos+neg):.2f}]")

Pool varyasyonlari:
  P0_KANSER_ONLY           : KANSER train only (192) [pos=132, neg=60, ratio=0.69]
  P1_BALANCED_625          : MASTER balanced 625P/625B (1250) [pos=625, neg=625, ratio=0.50]
  P2_KANSER_BAL625         : MASTER 625/625 + KANSER train (~1442) [pos=757, neg=685, ratio=0.52]
  P3_COMBINED_RAW          : COMBINED ham (3414) [pos=2549, neg=865, ratio=0.75]
  P4_COMBINED_KANSER       : COMBINED + KANSER train (3606) [pos=2681, neg=925, ratio=0.74]
  P5_FINAL_8020            : COMBINED %80B/%20P (1081) [pos=216, neg=865, ratio=0.20]
  P6_MASTER_UNDER          : MASTER undersample KANSER oraninda (388) [pos=268, neg=120, ratio=0.69]
  P7_MASTER_UNDER_K        : MASTER undersample + KANSER (580) [pos=400, neg=180, ratio=0.69]
  P8_COMBINED_BAL          : COMBINED balanced %50/%50 (1730) [pos=865, neg=865, ratio=0.50]
  P9_REVERSE_6040          : COMBINED %60B/%40P (1442) [pos=577, neg=865, ratio=0.40]


In [9]:
# Cell 9: Exp 0 — Finetune Duzeltme Dogrulama
print("="*70)
print("[Exp 0] Finetune Duzeltme Dogrulama (NB31 vs NB32)")
print("="*70)

all_results = {}
all_test_probas = {}

# NB31 referans degerleri
NB31_REF = {"E1_nn_ft": 0.5577, "E1_dnn_ft": 0.5129, "NB16_dnn_ft": 0.7046}

# Test 1: P1_BALANCED_625 ile finetune (NB16 recetesi)
# Test 2: P3_COMBINED_RAW ile finetune (Bug#4 izolasyonu)
for pool_name in ["P1_BALANCED_625", "P3_COMBINED_RAW"]:
    pool_df, pool_desc = build_pool(pool_name)
    # Preprocessor: pool + kanser_train birlesik basis (finetune icinde de kullanilir)
    pp_basis = pd.concat([pool_df, kanser_train], ignore_index=True)
    pp = fit_preprocessor(pp_basis)

    for kind, kind_label in [("nn", "nn_ft"), ("dnn", "dnn_ft")]:
        exp_key = f"E0_{pool_name}_{kind_label}"
        print(f"\n  {exp_key}: {pool_desc} -> KANSER finetune")
        try:
            oof, test_proba = oof_nn_finetune(kind, pool_df, kanser_train, kanser_test, pp)
            res = eval_model(exp_key, y_kanser_test, test_proba, y_kanser_train, oof)
            all_results[exp_key] = res
            all_test_probas[exp_key] = test_proba
            nb31_ref = NB31_REF.get(f"E1_{kind_label}", 0)
            delta = res["boot_f1"] - nb31_ref
            print(f"    Boot-F1={res['boot_f1']:.4f} +/- {res['boot_std']:.3f}  "
                  f"NB31 ref={nb31_ref:.4f}  delta={delta:+.4f}")
            print(f"    AUPRC={res['auprc']:.4f}  Prec={res['precision']:.3f}  "
                  f"Rec={res['recall']:.3f}  MCC={res['mcc']:.3f}  FP={res['fp']}  FN={res['fn']}")
        except Exception as e:
            print(f"    HATA: {e}")
            all_results[exp_key] = None

# Exp 0 ozet
print("\n" + "-"*70)
print("Exp 0 Ozet:")
print(f"{'Deney':<40s} {'Boot-F1':>8s} {'NB31 ref':>9s} {'Delta':>8s}")
print("-"*70)
for k, v in all_results.items():
    if v and k.startswith("E0_"):
        kind_label = k.split("_")[-1]
        nb31_ref = NB31_REF.get(f"E1_{kind_label}", 0)
        delta = v["boot_f1"] - nb31_ref
        print(f"{k:<40s} {v['boot_f1']:>8.4f} {nb31_ref:>9.4f} {delta:>+8.4f}")

[Exp 0] Finetune Duzeltme Dogrulama (NB31 vs NB32)

  E0_P1_BALANCED_625_nn_ft: MASTER balanced 625P/625B (1250) -> KANSER finetune
    Boot-F1=0.6018 +/- 0.055  NB31 ref=0.5577  delta=+0.0441
    AUPRC=0.9465  Prec=0.896  Rec=0.774  MCC=0.542  FP=12  FN=30

  E0_P1_BALANCED_625_dnn_ft: MASTER balanced 625P/625B (1250) -> KANSER finetune
    Boot-F1=0.6468 +/- 0.077  NB31 ref=0.5129  delta=+0.1339
    AUPRC=0.9339  Prec=0.929  Rec=0.692  MCC=0.533  FP=7  FN=41

  E0_P3_COMBINED_RAW_nn_ft: COMBINED ham (3414) -> KANSER finetune
    Boot-F1=0.6176 +/- 0.049  NB31 ref=0.5577  delta=+0.0599
    AUPRC=0.9043  Prec=0.895  Rec=0.835  MCC=0.597  FP=13  FN=22

  E0_P3_COMBINED_RAW_dnn_ft: COMBINED ham (3414) -> KANSER finetune
    Boot-F1=0.6213 +/- 0.045  NB31 ref=0.5129  delta=+0.1084
    AUPRC=0.9266  Prec=0.895  Rec=0.835  MCC=0.597  FP=13  FN=22

----------------------------------------------------------------------
Exp 0 Ozet:
Deney                                     Boot-F1  NB31 ref   

In [10]:
# Key parser: "E1_P3_COMBINED_RAW_nn_ft" -> (pool="P3_COMBINED_RAW", model="nn_ft")
_ALL_MODEL_NAMES = ["lgbm", "xgb", "catboost", "nn", "dnn", "nn_ft", "dnn_ft"]
def parse_exp_key(k):
    for mn in sorted(_ALL_MODEL_NAMES, key=len, reverse=True):
        suffix = "_" + mn
        if k.endswith(suffix):
            prefix = k[:len(k)-len(suffix)]
            pool = prefix.split("_", 1)[1] if "_" in prefix else prefix
            return pool, mn
    parts = k.split("_")
    return "_".join(parts[1:-1]), parts[-1]

# Cell 10: Exp 1 — Tam Pool x Model Sweep (ANA DENEY)
print("\n" + "="*70)
print("[Exp 1] Tam Pool x Model Sweep: 7 model x 10 pool")
print("="*70)

TREE_MODELS = ["lgbm", "xgb", "catboost"]
SCRATCH_NN = ["nn", "dnn"]
FINETUNE_NN = ["nn_ft", "dnn_ft"]
ALL_MODELS = TREE_MODELS + SCRATCH_NN + FINETUNE_NN

n_total = 0
n_done = 0
for pn in POOL_NAMES:
    for mt in ALL_MODELS:
        if pn == "P0_KANSER_ONLY" and mt in FINETUNE_NN:
            continue
        n_total += 1

print(f"Toplam {n_total} deney calistirilacak\n")

for pi, pool_name in enumerate(POOL_NAMES):
    pool_df, pool_desc = build_pool(pool_name)
    y_pool = pool_df[TARGET].values
    print(f"\n--- [{pi+1}/{len(POOL_NAMES)}] {pool_name}: {pool_desc} ---")

    # Preprocessor: pool uzerinde fit
    pp_pool = fit_preprocessor(pool_df)
    X_pool = transform_X(pool_df, pp_pool)
    X_test_pool = transform_X(kanser_test, pp_pool)

    for mt in ALL_MODELS:
        if pool_name == "P0_KANSER_ONLY" and mt in FINETUNE_NN:
            continue

        exp_key = f"E1_{pool_name}_{mt}"
        n_done += 1

        try:
            if mt in TREE_MODELS:
                if not HAS_CATBOOST and mt == "catboost":
                    print(f"  [{n_done}/{n_total}] {exp_key}: SKIP (catboost yok)")
                    continue
                oof, test_proba = oof_tree(mt, X_pool, y_pool, X_test_pool)
                # Tree/scratch: threshold pool OOF uzerinde, test kanser_test
                thr_y, thr_p = y_pool, oof

            elif mt in SCRATCH_NN:
                oof, test_proba = oof_nn_scratch(mt, pool_df, kanser_test, pp_pool)
                thr_y, thr_p = y_pool, oof

            elif mt in FINETUNE_NN:
                kind = "nn" if mt == "nn_ft" else "dnn"
                pp_ft = fit_preprocessor(pd.concat([pool_df, kanser_train], ignore_index=True))
                oof, test_proba = oof_nn_finetune(kind, pool_df, kanser_train, kanser_test, pp_ft)
                # Finetune OOF kanser_train boyutunda
                thr_y, thr_p = y_kanser_train, oof

            res = eval_model(exp_key, y_kanser_test, test_proba, thr_y, thr_p)
            all_results[exp_key] = res
            all_test_probas[exp_key] = test_proba
            print(f"  [{n_done}/{n_total}] {exp_key}: Boot-F1={res['boot_f1']:.4f} +/- {res['boot_std']:.3f}  "
                  f"Prec={res['precision']:.3f}  FP={res['fp']}  FN={res['fn']}  "
                  f"Train-F1={res['train_f1']:.3f}")

        except Exception as e:
            print(f"  [{n_done}/{n_total}] {exp_key}: HATA - {e}")
            import traceback; traceback.print_exc()
            all_results[exp_key] = None

    del pool_df
    gc.collect()

# Exp 1 ozet: en iyi 15
print("\n" + "="*70)
print("Exp 1 Top-15 Sonuclar:")
print(f"{'Deney':<45s} {'Boot-F1':>8s} {'Std':>6s} {'Prec':>6s} {'FP':>4s} {'FN':>4s}")
print("-"*75)
e1_sorted = sorted(
    [(k, v) for k, v in all_results.items() if v and k.startswith("E1_")],
    key=lambda x: x[1]["boot_f1"], reverse=True
)
for k, v in e1_sorted[:15]:
    print(f"{k:<45s} {v['boot_f1']:>8.4f} {v['boot_std']:>6.3f} "
          f"{v['precision']:>6.3f} {v['fp']:>4d} {v['fn']:>4d}")
print(f"\nNB31 E6 referans: Boot-F1=0.7201")


[Exp 1] Tam Pool x Model Sweep: 7 model x 10 pool
Toplam 68 deney calistirilacak


--- [1/10] P0_KANSER_ONLY: KANSER train only (192) ---
  [1/68] E1_P0_KANSER_ONLY_lgbm: Boot-F1=0.6347 +/- 0.063  Prec=0.912  FP=10  FN=29  Train-F1=0.820
  [2/68] E1_P0_KANSER_ONLY_xgb: Boot-F1=0.5895 +/- 0.083  Prec=0.928  FP=6  FN=56  Train-F1=0.757
  [3/68] E1_P0_KANSER_ONLY_catboost: Boot-F1=0.6666 +/- 0.066  Prec=0.933  FP=7  FN=35  Train-F1=0.810
  [4/68] E1_P0_KANSER_ONLY_nn: Boot-F1=0.4796 +/- 0.044  Prec=0.818  FP=25  FN=21  Train-F1=0.847
  [5/68] E1_P0_KANSER_ONLY_dnn: Boot-F1=0.5835 +/- 0.065  Prec=0.893  FP=12  FN=33  Train-F1=0.813

--- [2/10] P1_BALANCED_625: MASTER balanced 625P/625B (1250) ---
  [6/68] E1_P1_BALANCED_625_lgbm: Boot-F1=0.6577 +/- 0.066  Prec=0.933  FP=7  FN=35  Train-F1=0.717
  [7/68] E1_P1_BALANCED_625_xgb: Boot-F1=0.6872 +/- 0.052  Prec=0.927  FP=9  FN=19  Train-F1=0.761
  [8/68] E1_P1_BALANCED_625_catboost: Boot-F1=0.6958 +/- 0.049  Prec=0.933  FP=8  FN=22  Train-F1=

In [11]:
# Cell 11: Exp 2 — FE Etkilesim Testi
print("\n" + "="*70)
print("[Exp 2] FE Etkilesim Testi")
print("="*70)

# En iyi pool'u bul (Exp 1'den)
e1_valid = [(k, v) for k, v in all_results.items() if v and k.startswith("E1_")]
if e1_valid:
    best_e1_key, best_e1_val = max(e1_valid, key=lambda x: x[1]["boot_f1"])
    best_pool, best_model = parse_exp_key(best_e1_key)
    print(f"En iyi Exp 1: {best_e1_key} (Boot-F1={best_e1_val['boot_f1']:.4f})")
    print(f"En iyi pool: {best_pool}")

    # Top 3 model (farkli pool'larda en iyi olan modeller)
    model_best = {}
    for k, v in e1_valid:
        _, mt = parse_exp_key(k)
        if mt not in model_best or v["boot_f1"] > model_best[mt]:
            model_best[mt] = v["boot_f1"]
    top_models = sorted(model_best.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f"Top 3 model: {[m for m, _ in top_models]}")

    # En iyi pool ile no_fe vs with_fe
    pool_df, pool_desc = build_pool(best_pool)

    for mt, _ in top_models:
        for fe_mode in ["no_fe", "with_fe"]:
            exp_key = f"E2_{best_pool}_{mt}_{fe_mode}"
            print(f"\n  {exp_key}:")
            try:
                if fe_mode == "with_fe":
                    pool_fe = add_fe(pool_df)
                    test_fe = add_fe(kanser_test)
                    train_fe = add_fe(kanser_train)
                    fe_keep = keep_cols + FE_NEW_COLS
                else:
                    pool_fe = pool_df
                    test_fe = kanser_test
                    train_fe = kanser_train
                    fe_keep = keep_cols

                pp_fe = fit_preprocessor(pool_fe, keep_cols=fe_keep)
                X_pool_fe = transform_X(pool_fe, pp_fe)
                X_test_fe = transform_X(test_fe, pp_fe)
                y_pool = pool_fe[TARGET].values

                if mt in TREE_MODELS:
                    oof, test_proba = oof_tree(mt, X_pool_fe, y_pool, X_test_fe)
                    thr_y, thr_p = y_pool, oof
                elif mt in SCRATCH_NN:
                    oof, test_proba = oof_nn_scratch(mt, pool_fe, test_fe, pp_fe)
                    thr_y, thr_p = y_pool, oof
                elif mt in FINETUNE_NN:
                    kind = "nn" if mt == "nn_ft" else "dnn"
                    pp_ft = fit_preprocessor(
                        pd.concat([pool_fe, train_fe], ignore_index=True), keep_cols=fe_keep)
                    oof, test_proba = oof_nn_finetune(kind, pool_fe, train_fe, test_fe, pp_ft)
                    thr_y, thr_p = y_kanser_train, oof

                res = eval_model(exp_key, y_kanser_test, test_proba, thr_y, thr_p)
                all_results[exp_key] = res
                all_test_probas[exp_key] = test_proba
                print(f"    Boot-F1={res['boot_f1']:.4f} +/- {res['boot_std']:.3f}  "
                      f"Prec={res['precision']:.3f}  FP={res['fp']}  FN={res['fn']}")
            except Exception as e:
                print(f"    HATA: {e}")
                import traceback; traceback.print_exc()
                all_results[exp_key] = None

    # FE ozet
    print("\n  FE Etkilesim Ozet:")
    for mt, _ in top_models:
        nofe_key = f"E2_{best_pool}_{mt}_no_fe"
        fe_key = f"E2_{best_pool}_{mt}_with_fe"
        nofe_v = all_results.get(nofe_key)
        fe_v = all_results.get(fe_key)
        if nofe_v and fe_v:
            delta = fe_v["boot_f1"] - nofe_v["boot_f1"]
            print(f"    {mt}: no_fe={nofe_v['boot_f1']:.4f}  with_fe={fe_v['boot_f1']:.4f}  delta={delta:+.4f}")
else:
    print("Exp 1 sonucu yok, Exp 2 atlaniyor.")


[Exp 2] FE Etkilesim Testi
En iyi Exp 1: E1_P9_REVERSE_6040_catboost (Boot-F1=0.7104)
En iyi pool: P9_REVERSE_6040
Top 3 model: ['catboost', 'lgbm', 'xgb']

  E2_P9_REVERSE_6040_catboost_no_fe:
    Boot-F1=0.7104 +/- 0.042  Prec=0.930  FP=9  FN=14

  E2_P9_REVERSE_6040_catboost_with_fe:
    Boot-F1=0.7300 +/- 0.033  Prec=0.933  FP=9  FN=8

  E2_P9_REVERSE_6040_lgbm_no_fe:
    Boot-F1=0.6601 +/- 0.057  Prec=0.918  FP=10  FN=21

  E2_P9_REVERSE_6040_lgbm_with_fe:
    Boot-F1=0.6578 +/- 0.058  Prec=0.917  FP=10  FN=22

  E2_P9_REVERSE_6040_xgb_no_fe:
    Boot-F1=0.6652 +/- 0.054  Prec=0.918  FP=10  FN=21

  E2_P9_REVERSE_6040_xgb_with_fe:
    Boot-F1=0.6876 +/- 0.044  Prec=0.921  FP=10  FN=16

  FE Etkilesim Ozet:
    catboost: no_fe=0.7104  with_fe=0.7300  delta=+0.0195
    lgbm: no_fe=0.6601  with_fe=0.6578  delta=-0.0023
    xgb: no_fe=0.6652  with_fe=0.6876  delta=+0.0223


In [12]:
# Cell 12: Exp 3 — Stacking (En Iyi Config)
print("\n" + "="*70)
print("[Exp 3] OOF Stacking (LGBM + CatBoost + en iyi finetune)")
print("="*70)

# Key parser: "E1_P3_COMBINED_RAW_nn_ft" -> (pool="P3_COMBINED_RAW", model="nn_ft")
_ALL_MODEL_NAMES = ["lgbm", "xgb", "catboost", "nn", "dnn", "nn_ft", "dnn_ft"]
def parse_exp_key(k):
    """Parse E1_POOLNAME_model key, multi-word model isimleri (nn_ft, dnn_ft) icin."""
    for mn in sorted(_ALL_MODEL_NAMES, key=len, reverse=True):
        suffix = "_" + mn
        if k.endswith(suffix):
            prefix = k[:len(k)-len(suffix)]
            pool = prefix.split("_", 1)[1] if "_" in prefix else prefix
            return pool, mn
    parts = k.split("_")
    return "_".join(parts[1:-1]), parts[-1]

# En iyi pool ve finetune model'i bul
e1_valid = [(k, v) for k, v in all_results.items() if v and k.startswith("E1_")]
if not e1_valid:
    print("Exp 1 sonucu yok, Exp 3 atlaniyor.")
else:
    pool_model_best = {}
    for k, v in e1_valid:
        pn, mt = parse_exp_key(k)
        if mt not in pool_model_best or v["boot_f1"] > pool_model_best[mt][1]:
            pool_model_best[mt] = (pn, v["boot_f1"])

    print("Model bazinda en iyi pool'lar:")
    for mt, (pn, f1) in sorted(pool_model_best.items(), key=lambda x: x[1][1], reverse=True):
        print(f"  {mt:10s}: {pn} (Boot-F1={f1:.4f})")

    # En iyi finetune model
    ft_candidates = {mt: v for mt, v in pool_model_best.items() if mt in ["nn_ft", "dnn_ft"]}
    if ft_candidates:
        best_ft_model = max(ft_candidates, key=lambda x: ft_candidates[x][1])
        best_ft_pool = ft_candidates[best_ft_model][0]
        print(f"\nEn iyi finetune: {best_ft_model} (pool={best_ft_pool}, F1={ft_candidates[best_ft_model][1]:.4f})")
    else:
        best_ft_model = None
        best_ft_pool = None
        print("\nFinetune model bulunamadi!")

    # Stacking icin pool: en cok tekrar eden en iyi pool
    from collections import Counter
    pool_counts = Counter([v[0] for v in pool_model_best.values()])
    stack_pool = pool_counts.most_common(1)[0][0]
    print(f"Stacking pool: {stack_pool}")

    # Stacking base'leri kanser_train uzerinde OOF uretmeli (finetune ile ayni boyut)
    print("\n  Tree base'leri kanser_train uzerinde OOF uretiyor...")
    pp_kan = fit_preprocessor(kanser_train)
    X_kan_tr = transform_X(kanser_train, pp_kan)
    X_kan_te = transform_X(kanser_test, pp_kan)

    # Base 1: LGBM
    print("  Base 1: LGBM OOF (kanser_train)...")
    oof_lgbm_kan, test_lgbm_kan = oof_tree("lgbm", X_kan_tr, y_kanser_train, X_kan_te)
    print(f"    OOF done. Test range: [{test_lgbm_kan.min():.3f}, {test_lgbm_kan.max():.3f}]")

    # Base 2: CatBoost
    print("  Base 2: CatBoost OOF (kanser_train)...")
    if HAS_CATBOOST:
        oof_cb_kan, test_cb_kan = oof_tree("catboost", X_kan_tr, y_kanser_train, X_kan_te)
    else:
        oof_cb_kan, test_cb_kan = oof_tree("xgb", X_kan_tr, y_kanser_train, X_kan_te)
    print(f"    OOF done. Test range: [{test_cb_kan.min():.3f}, {test_cb_kan.max():.3f}]")

    base_names = ["lgbm", "catboost"]
    oof_list = [oof_lgbm_kan, oof_cb_kan]
    test_list = [test_lgbm_kan, test_cb_kan]

    # Base 3: En iyi finetune (varsa)
    if best_ft_model:
        print(f"  Base 3: {best_ft_model} finetune (pool={best_ft_pool})...")
        ft_pool_df, _ = build_pool(best_ft_pool)
        pp_ft = fit_preprocessor(pd.concat([ft_pool_df, kanser_train], ignore_index=True))
        kind = "nn" if best_ft_model == "nn_ft" else "dnn"
        oof_ft, test_ft = oof_nn_finetune(kind, ft_pool_df, kanser_train, kanser_test, pp_ft)
        print(f"    OOF done. Test range: [{test_ft.min():.3f}, {test_ft.max():.3f}]")
        base_names.append(best_ft_model)
        oof_list.append(oof_ft)
        test_list.append(test_ft)

    # Meta-feature matrix
    oof_matrix = np.column_stack(oof_list)
    test_matrix = np.column_stack(test_list)
    meta_train = np.hstack([oof_matrix,
                            oof_matrix.mean(axis=1, keepdims=True),
                            oof_matrix.std(axis=1, keepdims=True)])
    meta_test = np.hstack([test_matrix,
                           test_matrix.mean(axis=1, keepdims=True),
                           test_matrix.std(axis=1, keepdims=True)])

    print(f"\n  Meta-feature: train={meta_train.shape}, test={meta_test.shape}")
    print(f"  Base models: {base_names}")

    # Meta-learner: LR (GBM meta DEGIL!)
    lr_meta = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, random_state=SEED)
    lr_meta.fit(meta_train, y_kanser_train)
    stack_train_proba = lr_meta.predict_proba(meta_train)[:, 1]
    stack_test_proba = lr_meta.predict_proba(meta_test)[:, 1]

    res_stack = eval_model("E3_stack_lr", y_kanser_test, stack_test_proba,
                           y_kanser_train, stack_train_proba)
    all_results["E3_stack_lr"] = res_stack
    all_test_probas["E3_stack_lr"] = stack_test_proba
    print(f"\n  E3_stack_lr: Boot-F1={res_stack['boot_f1']:.4f} +/- {res_stack['boot_std']:.3f}  "
          f"Prec={res_stack['precision']:.3f}  FP={res_stack['fp']}  FN={res_stack['fn']}")
    print(f"  NB31 E6 referans: 0.7201 | NB16 stack_lr: 0.716")


[Exp 3] OOF Stacking (LGBM + CatBoost + en iyi finetune)
Model bazinda en iyi pool'lar:
  catboost  : P9_REVERSE_6040 (Boot-F1=0.7104)
  lgbm      : P3_COMBINED_RAW (Boot-F1=0.7072)
  xgb       : P3_COMBINED_RAW (Boot-F1=0.7030)
  nn_ft     : P1_BALANCED_625 (Boot-F1=0.6694)
  dnn       : P2_KANSER_BAL625 (Boot-F1=0.6399)
  dnn_ft    : P9_REVERSE_6040 (Boot-F1=0.6091)
  nn        : P8_COMBINED_BAL (Boot-F1=0.6013)

En iyi finetune: nn_ft (pool=P1_BALANCED_625, F1=0.6694)
Stacking pool: P3_COMBINED_RAW

  Tree base'leri kanser_train uzerinde OOF uretiyor...
  Base 1: LGBM OOF (kanser_train)...
    OOF done. Test range: [0.000, 1.000]
  Base 2: CatBoost OOF (kanser_train)...
    OOF done. Test range: [0.003, 0.999]
  Base 3: nn_ft finetune (pool=P1_BALANCED_625)...
    OOF done. Test range: [0.005, 0.799]

  Meta-feature: train=(192, 5), test=(193, 5)
  Base models: ['lgbm', 'catboost', 'nn_ft']

  E3_stack_lr: Boot-F1=0.6491 +/- 0.099  Prec=0.953  FP=4  FN=51
  NB31 E6 referans: 0.7201

In [13]:
# Cell 13: Exp 4 — Ensemble Sweep (Exp3 + NB31 E6 baseline yeniden uretim)
print("\n" + "="*70)
print("[Exp 4] Ensemble Sweep: Exp3 stacking + NB31 E6 tarzı baseline")
print("="*70)

# NB31 E6 baseline'i yeniden uret: LGBM on COMBINED + multi-base ensemble
# NB31'de E6 = alpha*LGBM + (1-alpha)*stacking, biz burada COMBINED LGBM'i yeniden uretiyoruz
print("\n  NB31 E6 baseline yeniden uretiliyor (LGBM on COMBINED)...")
pp_comb = fit_preprocessor(df_combined)
X_comb = transform_X(df_combined, pp_comb)
X_test_comb = transform_X(kanser_test, pp_comb)
y_comb = df_combined[TARGET].values

# Full model LGBM on COMBINED -> kanser_test proba
from lightgbm import LGBMClassifier as _LC
_m = _LC(**LGBM_PARAMS)
_m.fit(X_comb, y_comb)
e6_base_proba = _m.predict_proba(X_test_comb)[:, 1]
print(f"  COMBINED LGBM test proba range: [{e6_base_proba.min():.3f}, {e6_base_proba.max():.3f}]")

# E3 stacking proba
e3_proba = all_test_probas.get("E3_stack_lr")
if e3_proba is None:
    print("  E3 stacking sonucu yok, Exp 4 atlaniyor.")
else:
    # Ensemble sweep
    alphas = [0.3, 0.4, 0.5, 0.6, 0.7]
    print(f"\n  Ensemble sweep: alpha * E3_stack + (1-alpha) * COMBINED_LGBM")
    print(f"  {'Alpha':<8s} {'Boot-F1':>8s} {'Std':>6s} {'Prec':>6s} {'FP':>4s} {'FN':>4s}")
    print("  " + "-"*45)

    for alpha in alphas:
        blend_proba = alpha * e3_proba + (1 - alpha) * e6_base_proba
        exp_key = f"E4_alpha_{alpha}"

        # Train proba icin de blend (threshold secimi)
        e3_train = all_test_probas.get("E3_stack_lr")
        # Train probasi olarak stacking train proba'yi kullan
        blend_train = stack_train_proba if 'stack_train_proba' in dir() else e3_proba

        res = eval_model(exp_key, y_kanser_test, blend_proba, y_kanser_train, blend_train)
        all_results[exp_key] = res
        all_test_probas[exp_key] = blend_proba
        print(f"  {alpha:<8.1f} {res['boot_f1']:>8.4f} {res['boot_std']:>6.3f} "
              f"{res['precision']:>6.3f} {res['fp']:>4d} {res['fn']:>4d}")

    # En iyi blend
    best_blend = max(
        [(k, v) for k, v in all_results.items() if v and k.startswith("E4_")],
        key=lambda x: x[1]["boot_f1"]
    )
    print(f"\n  En iyi blend: {best_blend[0]} (Boot-F1={best_blend[1]['boot_f1']:.4f})")
    print(f"  NB31 E6 referans: 0.7201")


[Exp 4] Ensemble Sweep: Exp3 stacking + NB31 E6 tarzı baseline

  NB31 E6 baseline yeniden uretiliyor (LGBM on COMBINED)...
  COMBINED LGBM test proba range: [0.004, 0.999]

  Ensemble sweep: alpha * E3_stack + (1-alpha) * COMBINED_LGBM
  Alpha     Boot-F1    Std   Prec   FP   FN
  ---------------------------------------------
  0.3        0.6827  0.045  0.919   10   19
  0.4        0.6520  0.059  0.915   10   26
  0.5        0.6808  0.059  0.929    8   29
  0.6        0.7253  0.072  0.952    5   34
  0.7        0.6881  0.079  0.949    5   40

  En iyi blend: E4_alpha_0.6 (Boot-F1=0.7253)
  NB31 E6 referans: 0.7201


In [14]:
# Cell 14: Sonuc Derleme + CSV + Gorsellestirmeler
print("\n" + "="*70)
print("SONUC DERLEMESI")
print("="*70)

# --- DataFrame ---
rows = []
for k, v in all_results.items():
    if v is None:
        continue
    exp = k.split("_")[0]
    if exp in ("E0", "E1", "E2"):
        pool, model = parse_exp_key(k)
    else:
        model = k
        pool = "-"
    rows.append({
        "experiment": k, "exp_group": exp, "pool": pool, "model": model,
        "boot_f1": v["boot_f1"], "boot_std": v["boot_std"],
        "boot_lo": v["boot_lo"], "boot_hi": v["boot_hi"],
        "auprc": v["auprc"], "precision": v["precision"], "recall": v["recall"],
        "mcc": v["mcc"], "fp": v["fp"], "fn": v["fn"],
        "thr": v["thr"], "train_f1": v["train_f1"],
        "overfit_gap": v["train_f1"] - v["boot_f1"]
    })

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values("boot_f1", ascending=False).reset_index(drop=True)

# CSV kaydet
csv_path = os.path.join(RESULTS_DIR, "pool_sweep_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"Sonuclar: {csv_path} ({len(results_df)} satir)")

# Finetune karsilastirma CSV
ft_rows = []
for k, v in all_results.items():
    if v and k.startswith("E0_"):
        ft_rows.append({"experiment": k, "boot_f1": v["boot_f1"], "boot_std": v["boot_std"],
                        "auprc": v["auprc"], "fp": v["fp"], "fn": v["fn"]})
if ft_rows:
    ft_df = pd.DataFrame(ft_rows)
    ft_csv = os.path.join(RESULTS_DIR, "finetune_comparison.csv")
    ft_df.to_csv(ft_csv, index=False)
    print(f"Finetune karsilastirma: {ft_csv}")

# Top 20
print("\n--- TOP 20 ---")
print(results_df[["experiment", "pool", "model", "boot_f1", "boot_std",
                   "precision", "fp", "fn", "train_f1", "overfit_gap"]].head(20).to_string(index=False))

# --- Figure 1: Pool x Model Heatmap ---
e1_df = results_df[results_df["exp_group"] == "E1"].copy()
if len(e1_df) > 0:
    pivot = e1_df.pivot_table(index="model", columns="pool", values="boot_f1", aggfunc="first")
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto",
                   vmin=pivot.values[~np.isnan(pivot.values)].min() - 0.02,
                   vmax=pivot.values[~np.isnan(pivot.values)].max() + 0.02)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([c.replace("P", "P") for c in pivot.columns], rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=7,
                        color="black" if 0.4 < val < 0.7 else "white")
    plt.colorbar(im, ax=ax, label="Boot-F1 (%80/20)")
    ax.set_title("Exp 1: Model x Pool Boot-F1 Heatmap")
    plt.tight_layout()
    fig1_path = os.path.join(RESULTS_DIR, "fig1_pool_heatmap.png")
    plt.savefig(fig1_path, dpi=150)
    plt.close()
    print(f"Fig1: {fig1_path}")

# --- Figure 2: Finetune Fix ---
e0_data = {k: v for k, v in all_results.items() if v and k.startswith("E0_")}
if e0_data:
    fig, ax = plt.subplots(figsize=(10, 5))
    labels = list(e0_data.keys())
    vals = [e0_data[k]["boot_f1"] for k in labels]
    stds = [e0_data[k]["boot_std"] for k in labels]
    x = range(len(labels))
    bars = ax.bar(x, vals, yerr=stds, capsize=4, color="steelblue", alpha=0.8)
    ax.axhline(y=0.5577, color="red", linestyle="--", label="NB31 E1_nn_ft (0.5577)")
    ax.axhline(y=0.7046, color="green", linestyle="--", label="NB16 dnn_ft (0.7046)")
    ax.axhline(y=0.7201, color="orange", linestyle="--", label="NB31 E6 (0.7201)")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=7)
    ax.set_ylabel("Boot-F1 (%80/20)")
    ax.set_title("Exp 0: Finetune Duzeltme Dogrulama")
    ax.legend(fontsize=7)
    plt.tight_layout()
    fig2_path = os.path.join(RESULTS_DIR, "fig2_finetune_fix.png")
    plt.savefig(fig2_path, dpi=150)
    plt.close()
    print(f"Fig2: {fig2_path}")

# --- Figure 3: FE Interaction ---
e2_data = {k: v for k, v in all_results.items() if v and k.startswith("E2_")}
if e2_data:
    fig, ax = plt.subplots(figsize=(8, 5))
    labels_e2 = list(e2_data.keys())
    vals_e2 = [e2_data[k]["boot_f1"] for k in labels_e2]
    colors = ["#4CAF50" if "with_fe" in k else "#2196F3" for k in labels_e2]
    ax.bar(range(len(labels_e2)), vals_e2, color=colors, alpha=0.8)
    ax.set_xticks(range(len(labels_e2)))
    ax.set_xticklabels(labels_e2, rotation=30, ha="right", fontsize=7)
    ax.set_ylabel("Boot-F1 (%80/20)")
    ax.set_title("Exp 2: FE Etkilesim (yesil=with_fe, mavi=no_fe)")
    plt.tight_layout()
    fig3_path = os.path.join(RESULTS_DIR, "fig3_fe_interaction.png")
    plt.savefig(fig3_path, dpi=150)
    plt.close()
    print(f"Fig3: {fig3_path}")

# --- Figure 4: Stacking + Ensemble ---
e34_data = {k: v for k, v in all_results.items() if v and (k.startswith("E3_") or k.startswith("E4_"))}
if e34_data:
    fig, ax = plt.subplots(figsize=(8, 5))
    labels_e34 = sorted(e34_data.keys())
    vals_e34 = [e34_data[k]["boot_f1"] for k in labels_e34]
    stds_e34 = [e34_data[k]["boot_std"] for k in labels_e34]
    ax.bar(range(len(labels_e34)), vals_e34, yerr=stds_e34, capsize=4, color="teal", alpha=0.8)
    ax.axhline(y=0.7201, color="orange", linestyle="--", label="NB31 E6 (0.7201)")
    ax.set_xticks(range(len(labels_e34)))
    ax.set_xticklabels(labels_e34, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Boot-F1 (%80/20)")
    ax.set_title("Exp 3-4: Stacking + Ensemble Sweep")
    ax.legend()
    plt.tight_layout()
    fig4_path = os.path.join(RESULTS_DIR, "fig4_stacking_ensemble.png")
    plt.savefig(fig4_path, dpi=150)
    plt.close()
    print(f"Fig4: {fig4_path}")

# --- Figure 5: Best Confusion Matrix ---
if len(results_df) > 0:
    best_key = results_df.iloc[0]["experiment"]
    best_v = all_results[best_key]
    fig, ax = plt.subplots(figsize=(5, 4))
    cm = np.array([[best_v["tn"], best_v["fp"]], [best_v["fn"], best_v["tp"]]])
    im = ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=16, fontweight="bold")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred Benign", "Pred Patho"])
    ax.set_yticklabels(["True Benign", "True Patho"])
    ax.set_title(f"Best: {best_key}\nBoot-F1={best_v['boot_f1']:.4f}")
    plt.tight_layout()
    fig5_path = os.path.join(RESULTS_DIR, "fig5_best_confusion.png")
    plt.savefig(fig5_path, dpi=150)
    plt.close()
    print(f"Fig5: {fig5_path}")

print("\nTum gorseller ve CSV dosyalari kaydedildi.")


SONUC DERLEMESI
Sonuclar: /Users/tefe/teknofest_model/teknofest_model/results/v17_pretrain_distribution/pool_sweep_results.csv (84 satir)
Finetune karsilastirma: /Users/tefe/teknofest_model/teknofest_model/results/v17_pretrain_distribution/finetune_comparison.csv

--- TOP 20 ---
                         experiment                          pool        model  boot_f1  boot_std  precision  fp  fn  train_f1  overfit_gap
E2_P9_REVERSE_6040_catboost_with_fe P9_REVERSE_6040_catboost_with           fe 0.729970  0.033455   0.932836   9   8  0.733728     0.003758
                       E4_alpha_0.6                             - E4_alpha_0.6 0.725342  0.072417   0.951923   5  34  0.774775     0.049433
  E2_P9_REVERSE_6040_catboost_no_fe   P9_REVERSE_6040_catboost_no           fe 0.710438  0.041621   0.929688   9  14  0.720000     0.009562
        E1_P9_REVERSE_6040_catboost               P9_REVERSE_6040     catboost 0.710438  0.041621   0.929688   9  14  0.720000     0.009562
            E1_P3_C

In [15]:
# Cell 15: PDF Rapor
from fpdf import FPDF
from PIL import Image

class NB32Report(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 6, "NB32 - KANSER Training Pool Sweep + Finetune Fix", align="C", new_x="LMARGIN", new_y="NEXT")
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)
    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", align="C")
    def section(self, title):
        self.set_font("Helvetica", "B", 12)
        self.cell(0, 8, title, new_x="LMARGIN", new_y="NEXT")
        self.ln(2)
    def body_text(self, txt):
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, txt)
        self.ln(2)
    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [190 // len(headers)] * len(headers)
        self.set_font("Helvetica", "B", 8)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), border=1, align="C")
        self.ln()
        self.set_font("Helvetica", "", 7)
        for row in rows:
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), border=1, align="C")
            self.ln()
        self.ln(3)
    def add_image_safe(self, path, w=170):
        if os.path.exists(path):
            self.image(path, x=20, w=w)
            self.ln(5)

pdf = NB32Report()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=20)

# Sayfa 1: Baslik
pdf.add_page()
pdf.set_font("Helvetica", "B", 16)
pdf.cell(0, 15, "NB32: KANSER Training Pool Sweep", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 8, f"Tarih: {datetime.now().strftime('%Y-%m-%d %H:%M')}", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.ln(10)
pdf.body_text(
    "Bu rapor, KANSER paneli icin 7 farkli model tipi x 10 farkli training pool "
    "varyasyonunu sistematik olarak test eder. NB31'deki finetune implementasyon hatalari "
    "duzeltilmis, NB16'nin kanitlanmis _train_es recetesi tum NN/DNN modellere uygulanmistir.\n\n"
    f"Baseline: NB31 E6_alpha_0.5 Boot-F1 = 0.7201\n"
    f"Toplam deney sayisi: {len(results_df)}"
)

# Sayfa 2: Exp 0 Finetune Fix
pdf.add_page()
pdf.section("1. Exp 0: Finetune Duzeltme Dogrulama")
pdf.body_text(
    "NB31'deki 4 implementasyon hatasi duzeltildi:\n"
    "1) Layer freeze: Sadece son layer -> Tum katmanlar lr=1e-4\n"
    "2) Scaler: Her fold yeni -> Birlesik basis (pretrain+panel)\n"
    "3) Early stopping: Loss-based -> F1-based\n"
    "4) weight_decay: 1e-4 -> 1e-3, pos_weight eklendi"
)
e0_rows = []
for k, v in sorted(all_results.items()):
    if v and k.startswith("E0_"):
        e0_rows.append([k, f"{v['boot_f1']:.4f}", f"{v['boot_std']:.3f}",
                        f"{v['auprc']:.4f}", str(v['fp']), str(v['fn'])])
if e0_rows:
    pdf.add_table(["Deney", "Boot-F1", "Std", "AUPRC", "FP", "FN"],
                  e0_rows, [55, 25, 20, 25, 15, 15])
fig2_p = os.path.join(RESULTS_DIR, "fig2_finetune_fix.png")
pdf.add_image_safe(fig2_p)

# Sayfa 3: Exp 1 Pool Sweep
pdf.add_page()
pdf.section("2. Exp 1: Training Pool x Model Sweep")
pdf.body_text(
    "7 model tipi (LGBM, XGB, CatBoost, SmallMLP, DeepMLP, NN_ft, DNN_ft) x "
    "10 training pool varyasyonu test edildi."
)
fig1_p = os.path.join(RESULTS_DIR, "fig1_pool_heatmap.png")
pdf.add_image_safe(fig1_p)

# Top 10
pdf.section("Top 10 Sonuclar")
top10 = results_df[results_df["exp_group"]=="E1"].head(10)
t10_rows = []
for _, r in top10.iterrows():
    t10_rows.append([r["experiment"][:40], f"{r['boot_f1']:.4f}", f"{r['boot_std']:.3f}",
                     f"{r['precision']:.3f}", str(int(r['fp'])), str(int(r['fn'])),
                     f"{r['train_f1']:.3f}"])
if t10_rows:
    pdf.add_table(["Deney", "Boot-F1", "Std", "Prec", "FP", "FN", "Train-F1"],
                  t10_rows, [55, 22, 18, 22, 12, 12, 22])

# Sayfa 4: Exp 2 FE
pdf.add_page()
pdf.section("3. Exp 2: Feature Engineering Etkilesimi")
fig3_p = os.path.join(RESULTS_DIR, "fig3_fe_interaction.png")
pdf.add_image_safe(fig3_p)

# Sayfa 5: Exp 3-4 Stacking + Ensemble
pdf.add_page()
pdf.section("4. Exp 3-4: Stacking + Ensemble Sweep")
e34_rows = []
for k in sorted(all_results.keys()):
    v = all_results[k]
    if v and (k.startswith("E3_") or k.startswith("E4_")):
        e34_rows.append([k, f"{v['boot_f1']:.4f}", f"{v['boot_std']:.3f}",
                        f"{v['precision']:.3f}", str(v['fp']), str(v['fn'])])
if e34_rows:
    pdf.add_table(["Deney", "Boot-F1", "Std", "Prec", "FP", "FN"],
                  e34_rows, [45, 25, 20, 25, 15, 15])
fig4_p = os.path.join(RESULTS_DIR, "fig4_stacking_ensemble.png")
pdf.add_image_safe(fig4_p)

# Sayfa 6: Best + CM
pdf.add_page()
pdf.section("5. En Iyi Model ve Confusion Matrix")
if len(results_df) > 0:
    best = results_df.iloc[0]
    pdf.body_text(
        f"En iyi: {best['experiment']}\n"
        f"Boot-F1: {best['boot_f1']:.4f} +/- {best['boot_std']:.3f}\n"
        f"Precision: {best['precision']:.3f}, FP={int(best['fp'])}, FN={int(best['fn'])}\n"
        f"MCC: {best['mcc']:.3f}, AUPRC: {best['auprc']:.3f}\n"
        f"NB31 E6 referans: 0.7201"
    )
fig5_p = os.path.join(RESULTS_DIR, "fig5_best_confusion.png")
pdf.add_image_safe(fig5_p)

# Kaydet
pdf_path = os.path.join(REPORTS_DIR, "NB32_pretrain_distribution_report.pdf")
pdf.output(pdf_path)
print(f"PDF rapor: {pdf_path}")
print("NB32 tamamlandi!")

PDF rapor: /Users/tefe/teknofest_model/teknofest_model/reports/NB32_pretrain_distribution_report.pdf
NB32 tamamlandi!
